# Stage 3 — Sentiment Analysis (NLP Models)

Fine-tunes every model in `NLP_MODELS` on the weak-labeled AMZN headlines
from Stage 1, repeated across every seed in `CONFIG["random_seeds"]`. The
full sweep is `len(NLP_MODELS) × len(random_seeds)` runs (currently 4 × 10 = 40),
which is enough to compute mean ± std and run statistical significance tests
in Stage 6.

## What this stage produces
- `./results/nlp_results.csv` — one row per (model, seed) with accuracy, F1,
  train/inference times, and ISO 8601 UTC start/end timestamps for both phases.
- `./artifacts/nlp_probs.parquet` — per-headline 3-class probabilities from the
  single best (model, seed) combination, used by Stage 4 to build rolling
  sentiment features for the regression `+NLP` variant.
- `./artifacts/nlp_probs_run_meta.json` — sidecar metadata describing which
  run produced the probability file and its wall-clock timing windows.

## What this stage consumes
- `./artifacts/text_df.parquet` (weak-labeled headlines, from Stage 1).
- `./artifacts/num_df.parquet` (only used to compute the shared cutoff date).

## Energy join hook
Training and inference are bracketed with both `time.time()` (for fast
per-run summaries) and `datetime.now(timezone.utc).isoformat()` (for joining
against an external power log post-run). Energy is **not** measured by Python
here — a separate wall-meter / RAPL log is integrated over each run's
`[wall_*_start_iso, wall_*_end_iso]` window after the pipeline finishes.

## Best-model selection
The "best" run is picked by **F1, not accuracy**, because the weak labels are
class-imbalanced (mostly neutral) and accuracy can be inflated by always
predicting the majority class. F1 (weighted) penalizes that failure mode.


In [ ]:
# Load shared helpers, config values, and model registries.
from common import *
from datetime import datetime, timezone

# Hugging Face training utilities used in this stage.
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    GPT2ForSequenceClassification,
    GPT2Tokenizer,
    TrainingArguments,
    Trainer,
)

# text_df contains weak-labeled AMZN headlines from Stage 1.
text_df = load_text_df()
# num_df is loaded only to compute a shared split cutoff date.
num_df = load_num_df()

# Normalize timestamps to date-level precision for clean chronological splits.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# cutoff_date defines the train/test boundary used by both text and numeric tasks.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)
print(f"Chronological cutoff: {cutoff_date.date()}")

# Collector object accumulates per-run metrics for CSV export.
results = ResultsCollector()

## 3.1 Training routine

In [ ]:
def build_tokenizer_and_model(model_path, config):
    """Create tokenizer/model pair for a checkpoint path."""
    # GPT-2 special-case: the GPT-2 family is currently commented out of NLP_MODELS,
    # but this branch is preserved so that re-enabling GPT-2 is a one-line registry
    # change (just uncomment the entries in common.py). GPT-2 needs a pad token
    # (its tokenizer ships without one — we reuse EOS) and the corresponding
    # pad_token_id must be set on model.config so attention masks work correctly.
    # Treat this branch as dead-by-design, not dead-by-accident.
    is_gpt2 = "gpt2" in model_path
    if is_gpt2:
        tokenizer = GPT2Tokenizer.from_pretrained(model_path)
        tokenizer.pad_token = tokenizer.eos_token

        model = GPT2ForSequenceClassification.from_pretrained(
            model_path,
            num_labels=config["num_labels"],
        )
        model.config.pad_token_id = tokenizer.eos_token_id
    else:
        # Non-GPT models can use Auto classes directly.
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            num_labels=config["num_labels"],
        )

    # Move model to GPU/CPU selected in common.py.
    model.to(DEVICE)
    return tokenizer, model


def train_nlp_model(model_name, model_path, text_df, config, seed, cutoff_date):
    """Train one model for one seed and return metrics + predictions."""
    tokenizer, model = build_tokenizer_and_model(model_path=model_path, config=config)

    # Build chronological train/test datasets to avoid temporal leakage.
    train_ds, test_ds, _, _ = make_text_splits_chronological(
        df=text_df,
        tokenizer=tokenizer,
        max_length=config["max_length"],
        cutoff_date=cutoff_date,
        date_col="date",
    )

    # run_dir stores temporary trainer outputs for this run.
    run_dir = RESULTS_DIR / f"{model_name.replace(' ', '_')}_seed{seed}"
    # eval_strategy="epoch" runs the test-set evaluation at the end of each
    # training epoch, which gives us a learning curve and lets us catch
    # divergence early. Side-effect worth disclosing: the value reported in
    # `train_time_s` includes those per-epoch evaluations, not just gradient
    # updates. The energy-join numbers (wall_train_start_iso/end_iso) bracket
    # the same span, so external energy is consistent with reported time.
    training_args = TrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["train_batch_size"],
        per_device_eval_batch_size=config["eval_batch_size"],
        learning_rate=config["learning_rate"],
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        seed=seed,
        report_to="none",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=config.get("dataloader_num_workers", 0),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_clf_metrics,
    )

    # ISO timestamps are written so that an external power log (sampled by a
    # wall meter or RAPL outside this notebook) can be integrated over each
    # [start, end] window after the run completes. They are NOT used for
    # internal timing — `time.time()` deltas handle that. Both methods bracket
    # the same code so the windows agree to within a millisecond.
    wall_train_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    wall_train_end_iso = datetime.now(timezone.utc).isoformat()

    # Evaluate on chronological test split.
    eval_out = trainer.evaluate()

    # Measure prediction/inference time + wall-clock anchors for external energy join.
    wall_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    pred_out = trainer.predict(test_ds)
    infer_time = time.time() - t0
    wall_infer_end_iso = datetime.now(timezone.utc).isoformat()

    # Convert logits to class IDs via argmax.
    preds = np.argmax(pred_out.predictions, axis=-1)

    # record stores metrics that will be appended to nlp_results.csv.
    record = {
        "model": model_name,
        "seed": seed,
        "accuracy": eval_out["eval_accuracy"],
        "f1": eval_out["eval_f1"],
        "train_time_s": train_time,
        "infer_time_s": infer_time,
        "n_test_samples": len(test_ds),
        "wall_train_start_iso": wall_train_start_iso,
        "wall_train_end_iso": wall_train_end_iso,
        "wall_infer_start_iso": wall_infer_start_iso,
        "wall_infer_end_iso": wall_infer_end_iso,
    }

    # Free memory before the next model/seed run.
    del model, trainer
    torch.cuda.empty_cache()

    return record, preds


def train_best_model_and_export_probs(model_name, model_path, seed, text_df, cutoff_date):
    """Retrain best run and export per-headline class probabilities."""
    # We retrain the best (model, seed) from scratch instead of reusing a model
    # already in memory because the main sweep above issues `del model, trainer`
    # at the end of every run to free GPU memory between configurations. By the
    # time we know which run "won" (post-sweep), the winning weights are gone.
    #
    # Cost: ~10% extra compute relative to the main sweep (one extra training
    # of the best configuration). This is flagged as a future optimization;
    # the simplest fix is to checkpoint each trained model to disk during the
    # sweep and reload the winner here.
    tokenizer, model = build_tokenizer_and_model(model_path=model_path, config=CONFIG["nlp"])

    # Fit using chronological training slice only.
    train_ds, _, _, _ = make_text_splits_chronological(
        df=text_df,
        tokenizer=tokenizer,
        max_length=CONFIG["nlp"]["max_length"],
        cutoff_date=cutoff_date,
        date_col="date",
    )

    output_dir = RESULTS_DIR / f"best_{model_name.replace(' ', '_')}_seed{seed}"
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=CONFIG["nlp"]["epochs"],
        per_device_train_batch_size=CONFIG["nlp"]["train_batch_size"],
        per_device_eval_batch_size=CONFIG["nlp"]["eval_batch_size"],
        learning_rate=CONFIG["nlp"]["learning_rate"],
        save_strategy="no",
        seed=seed,
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=CONFIG["nlp"].get("dataloader_num_workers", 0),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
    )
    wall_best_train_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    trainer.train()
    best_train_time = time.time() - t0
    wall_best_train_end_iso = datetime.now(timezone.utc).isoformat()

    # Score EVERY headline (train slice + test slice), not just test. Stage 4's
    # `+NLP` variant builds rolling-K-day sentiment features by averaging
    # per-headline probabilities over a lookback window, which means it needs a
    # probability for every headline date in the analysis window — including
    # train-period headlines that contribute to the rolling average for the
    # earliest test-period rows. Labels are placeholders (zeros) because we
    # only consume the model's softmax output here, never compare to truth.
    infer_ds = SentimentDataset(
        texts=text_df["text"],
        labels=pd.Series(np.zeros(len(text_df), dtype=int)),
        tokenizer=tokenizer,
        max_length=CONFIG["nlp"]["max_length"],
    )

    wall_best_infer_start_iso = datetime.now(timezone.utc).isoformat()
    t0 = time.time()
    prediction_output = trainer.predict(infer_ds)
    best_infer_time = time.time() - t0
    wall_best_infer_end_iso = datetime.now(timezone.utc).isoformat()
    logits = prediction_output.predictions
    # Softmax converts logits into class probabilities summing to 1.
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

    # Save one row per headline so Stage 4 can build rolling sentiment features.
    nlp_probs_df = pd.DataFrame(
        {
            "date": text_df["date"].values,
            "prob_negative": probs[:, LABEL_TO_ID["negative"]],
            "prob_neutral": probs[:, LABEL_TO_ID["neutral"]],
            "prob_positive": probs[:, LABEL_TO_ID["positive"]],
            "model": model_name,
            "seed": seed,
        }
    )

    # Sidecar metadata file. The energy-join script reads this to know which
    # (model, seed) produced nlp_probs.parquet and over what wall-clock windows
    # the best-model retrain + scoring happened, so its energy can be attributed
    # separately from the main 40-run sweep recorded in nlp_results.csv.
    import json
    best_run_meta = {
        "model": model_name,
        "seed": seed,
        "wall_best_train_start_iso": wall_best_train_start_iso,
        "wall_best_train_end_iso": wall_best_train_end_iso,
        "best_train_time_s": best_train_time,
        "wall_best_infer_start_iso": wall_best_infer_start_iso,
        "wall_best_infer_end_iso": wall_best_infer_end_iso,
        "best_infer_time_s": best_infer_time,
        "n_headlines_scored": int(len(text_df)),
    }
    with open(ARTIFACTS_DIR / "nlp_probs_run_meta.json", "w") as f:
        json.dump(best_run_meta, f, indent=2)

    # Free memory after export.
    del model, trainer
    torch.cuda.empty_cache()

    return nlp_probs_df

## 3.2 Run every (model, seed) combination

In [ ]:
# nlp_predictions: built but never persisted and never read downstream. Kept for
# optional in-notebook diagnostics (e.g., confusion matrices, error analysis on
# specific seeds). Safe to remove if unused; kept here so a future reader can
# inspect raw predictions without re-running the sweep.
nlp_predictions = {}

# Train every model across every configured seed.
for model_name, model_path in NLP_MODELS.items():
    for seed in CONFIG["random_seeds"]:
        print(f"\n{'=' * 60}")
        print(f"  {model_name}  |  seed {seed}")
        print(f"{'=' * 60}")

        # Train and evaluate one (model, seed) run.
        record, preds = train_nlp_model(
            model_name=model_name,
            model_path=model_path,
            text_df=text_df,
            config=CONFIG["nlp"],
            seed=seed,
            cutoff_date=cutoff_date,
        )

        # Save metrics and predictions for this run.
        results.add_nlp(record)
        nlp_predictions[(model_name, seed)] = preds

        # Print compact run summary.
        print(f"  Accuracy: {record['accuracy']:.4f}  |  F1: {record['f1']:.4f}")
        print(
            f"  Train: {record['train_time_s']:.1f}s  |  "
            f"Infer: {record['infer_time_s']:.1f}s"
        )

# Preview collected NLP metrics.
results.nlp_df().round(6)

## 3.3 Persist results and export best-model headline probabilities

In [ ]:
# Save Stage 3 metrics to results/nlp_results.csv.
results.save(RESULTS_DIR)

# Identify the best run by F1 so Stage 4 can consume a single probability file.
nlp_results_df = results.nlp_df().copy()
best_row = nlp_results_df.sort_values("f1", ascending=False).iloc[0]

# Extract model ID and seed of the top run.
best_model_name = best_row["model"]
best_seed = int(best_row["seed"])
best_model_path = NLP_MODELS[best_model_name]

print("Best NLP run used for probability export:")
print(best_row.to_string())

# Re-train that best configuration and export per-headline probabilities.
nlp_probs_df = train_best_model_and_export_probs(
    model_name=best_model_name,
    model_path=best_model_path,
    seed=best_seed,
    text_df=text_df,
    cutoff_date=cutoff_date,
)
save_nlp_probs_df(nlp_probs_df)

print(f"Total NLP experiments: {len(results.nlp_results)}")
print(f"Saved per-headline probabilities to {ARTIFACTS_DIR / 'nlp_probs.parquet'}")